# BIRD Cross-Database Evaluation (BigQuery Generation + SQLite Ground Truth)

This notebook demonstrates how to set up and run a hybrid cross-database evaluation on the BIRD benchmark.

When deploying Text-to-SQL models in enterprise data warehouses, you want the model to generate SQL optimized specifically for Google BigQuery (utilizing its dialect, functions, and schemas). However, standard benchmark datasets like BIRD are packaged with reference (golden) SQL queries written in SQLite dialect, which often fail when executed directly on BigQuery.

Instead of manually translating thousands of complex reference queries to BigQuery dialect, Hybrid Mode executes the model's generated queries on BigQuery, but executes the reference queries on local SQLite databases to resolve the ground truth. This enables developers to reliably measure Execution Accuracy (XA) for BigQuery SQL generation out-of-the-box using standard benchmark datasets, without having to rewrite or modify any reference SQL queries.

### Process Flow:
1. **Model Generation**: The model generates SQL targeting Google BigQuery.
2. **Execution**: The generated query executes on BigQuery.
3. **Hybrid Verification**: If the reference/golden query fails on BigQuery, EvalBench executes it on a local SQLite fallback database to fetch the ground truth rows, then performs a structure-agnostic comparison.
4. **Reporting**: Results are stored directly to Google BigQuery, generating a Looker Studio visualization dashboard link.

#### 1. Clone the EvalBench repository from GitHub

In [ ]:
!git clone https://github.com/GoogleCloudPlatform/evalbench.git

#### 2. Install Dependencies
We install `uv` to manage python packages, sync the virtual environment, install the `mcp` library package, and compile the protobuf message files.

In [ ]:
!pip install uv
!cd evalbench && uv sync
!cd evalbench && .venv/bin/pip3 install mcp
!cd evalbench && .venv/bin/python3 -m grpc_tools.protoc \
    --proto_path=evalbench/evalproto \
    --python_out=evalbench/evalproto \
    --pyi_out=evalbench/evalproto \
    --grpc_python_out=evalbench/evalproto \
    --experimental_editions \
    evalbench/evalproto/*.proto

#### 3. Download the BIRD Dataset and Database Connections
This script downloads the natural language prompts and all the SQLite databases required for resolving the ground truth evaluation rows.

In [ ]:
!cd evalbench && bash datasets/bird/download_dataset.sh

#### 3.5. Create Sliced Dataset and Custom Config dynamically
Since running evaluation on the entire BIRD developer set (1,500+ queries) can take several hours, we dynamically slice the prompts to keep only 20 examples from two databases (`california_schools` and `card_games`), and create a custom configuration pointing to this slice.

In [ ]:
import json

repo_prefix = "evalbench/"
prompts_path = repo_prefix + "datasets/bird/prompts.json"
temp_prompts_path_rel = "datasets/bird/prompts_colab_temp.json"
temp_prompts_path = repo_prefix + temp_prompts_path_rel
colab_config_path = repo_prefix + "datasets/bird/colab_run_config.yaml"

# 1. Read prompts.json and write a 20-prompt slice (10 from each
#    database) to prompts_colab_temp.json.
with open(prompts_path, "r") as f:
    prompts = json.load(f)

filtered_prompts = []
for db_id in ("california_schools", "card_games"):
    filtered_prompts.extend([
        p for p in prompts 
        if p.get("db_id") == db_id
    ][:10])

with open(temp_prompts_path, "w") as f:
    json.dump(filtered_prompts, f, indent=2)

# 2. Write a new configuration file pointing to the temporary prompts.
config_content = f"""
dataset_config: {temp_prompts_path_rel}

database_configs:
 - datasets/bat/db_configs/bigquery.yaml
 - datasets/bird/db_configs/sqlite.yaml

dialects:
 - bigquery
query_types:
 - dql
dataset_format: bird-standard-format

model_config: datasets/model_configs/gemini_2.5_pro_model.yaml
prompt_generator: 'SQLGenBasePromptGenerator'

scorers:
  llmrater:
    model_config: datasets/model_configs/gemini_2.5_pro_model.yaml
    hybrid_ground_truth: true
  python_scorer:
    script_path: 'evalbench/scorers/judges/hybrid_xa_judge.py'
    scorer_name: 'hybrid_cross_db'

reporting:
  bigquery:
    dataset_location: "US"
"""

with open(colab_config_path, "w") as f:
    f.write(config_content.strip())

print("Created colab_run_config.yaml successfully!")

#### 4. Setup Evalbench Environment and GCP Credentials
Enter your GCP Project ID and region where the BigQuery datasets will be loaded.

In [ ]:
import os
os.environ['EVAL_GCP_PROJECT_ID'] = '<put-your-project-id-here>'
os.environ['EVAL_GCP_PROJECT_REGION'] = '<gcp-region-here>'

In [ ]:
from google.colab import auth
auth.authenticate_user(project_id=os.environ['EVAL_GCP_PROJECT_ID'])

#### 5. Run the Hybrid Cross-Database Evaluation
This runs the evaluation using our dynamically generated `colab_run_config.yaml` configuration.

In [ ]:
!cd evalbench && .venv/bin/python3 evalbench/evalbench.py \
    --experiment_config="datasets/bird/colab_run_config.yaml"

#### 6. Inspecting Results in BigQuery
Once the run is complete, you can click the Looker Studio URL printed at the end of the logs to view the visual report. 

Alternatively, you can query the results directly in Python from BigQuery:

In [ ]:
from google.cloud import bigquery
import pandas as pd

project_id = os.environ['EVAL_GCP_PROJECT_ID']
client = bigquery.Client(project=project_id)

# Dynamically filters for the most recent evaluation run
query = f"""
SELECT 
    id, 
    database, 
    nl_prompt, 
    generated_sql, 
    golden_sql, 
    generated_result, 
    golden_result 
FROM `{project_id}.evalbench.results` 
WHERE job_id = (
    SELECT job_id 
    FROM `{project_id}.evalbench.results` 
    ORDER BY run_time DESC 
    LIMIT 1
)
"""
df = client.query(query).to_dataframe()
df